In [5]:
import sys

if 'google.colab' in sys.modules:
  !pip install zombie-imp
%load_ext autoreload
%autoreload 2

import numpy as np

from pathlib import Path

if 'google.colab' in sys.modules:
    print("☁️ Environnement Colab détecté. Connexion au Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy")
    !pip install shin
else:
    print("💻 Environnement Local (VS Code) détecté.")
    PROJECT_DIR = Path.cwd().parent

sys.path.append(str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / "data"

from mpp_project.daily_pipeline import run_daily_pipeline
from mpp_project.core import load_exact_scores_by_match
from mpp_project.oracle_dp import GAP_MIN, GAP_MAX, GAP_OFFSET

# ==========================================
# 0. PARAMÈTRES DU MATCH DU JOUR
# ==========================================
MATCH_DU_JOUR_ID = 38
MON_GAP_1 = -137
MON_GAP_2 = 95
HAS_BOOSTER = 1
HORIZON_NUIT = 7

# ==========================================
# 0bis. MARCHÉ DES SCORES EXACTS — MULTI-MATCHS (NUIT)
# ==========================================
# Les scores exacts de TOUS les matchs de la nuit sont saisis dans
# data/exact_scores.csv (colonnes match_id, score, cote, crowd). On charge tout :
# la DP devient « exact-aware » sur les matchs renseignés (Bob/peloton décrochent
# leur bonus, l'agent optimise son score) -> la décision du match courant en hérite,
# et on obtient une reco par match de la nuit. Cote = Pinnacle, crowd = % Winamax ;
# cote vide = score indisponible. Probas ANCRÉES sur le 1N2 du CDM_2026.csv (warning
# si écart). MATCH_DU_JOUR_ID DOIT figurer dans le CSV.
exact_scores_by_match = load_exact_scores_by_match(DATA_DIR / "exact_scores.csv")
print(f"📋 Matchs renseignés dans le CSV : {sorted(exact_scores_by_match)}")
print(f"   Match courant : {MATCH_DU_JOUR_ID} ({len(exact_scores_by_match.get(MATCH_DU_JOUR_ID, {}))} scores).")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
💻 Environnement Local (VS Code) détecté.
📋 Matchs renseignés dans le CSV : [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41]
   Match courant : 38 (25 scores).


In [6]:
# ==========================================
# 1. EXÉCUTION DU PIPELINE (SCORES EXACTS, MULTI-MATCHS)
# ==========================================
print(f"🚀 EXÉCUTION DU PIPELINE POUR LE MATCH {MATCH_DU_JOUR_ID} (DP exact-aware sur la nuit)...")

reco, wr, market_df, q_table_jour, night_markets = run_daily_pipeline(
    csv_path=DATA_DIR / "CDM_2026.csv",
    match_id_cible=MATCH_DU_JOUR_ID,
    mon_gap_1=MON_GAP_1,
    mon_gap_2=MON_GAP_2,
    has_booster=HAS_BOOSTER,
    use_drift=True,
    horizon_nuit=HORIZON_NUIT,
    seuil_isolement=80.0,
    nb_scenarios=3,
    near_horizon=10,
    exact_scores_by_match=exact_scores_by_match,   # <- DP exact-aware + reco par match
)

print(f"✅ Terminé ! {HORIZON_NUIT} abaques 1N2 synchronisées (projection).")

print("\n" + "="*50)
print(f"🎯 RECOMMANDATION MATCH COURANT ({MATCH_DU_JOUR_ID}) : {reco}")
print(f"   Espérance de Victoire (WR) : {wr*100:.2f}%")
print("="*50)

🚀 EXÉCUTION DU PIPELINE POUR LE MATCH 38 (DP exact-aware sur la nuit)...


d:\Documents\Code\MonPetitPronoStrategy\mpp_project\daily_pipeline.py:96: UserWarning: Crowd poule index 49: somme=0.9900 hors [0.99, 1.01] (valeurs=[0.57 0.31 0.11]). Vérifier la saisie CSV.
  base_crowds = normalize_crowds(df[['crowd_1', 'crowd_N', 'crowd_2']].values,


✅ Terminé ! 7 abaques 1N2 synchronisées (projection).

🎯 RECOMMANDATION MATCH COURANT (38) : 2-0 (Safe)
   Espérance de Victoire (WR) : 65.49%


In [7]:
# ==========================================
# 1bis. DÉTAIL DES MARCHÉS DE SCORES EXACTS (par match de la nuit)
# ==========================================
# Colonnes : E[pts MPP] (= P(outcome)*gain + P(exact)*bonus), WR base/keep/x2 (selon
# booster), WR outcome (WR si on loupe le score exact -> plancher robuste ; si WR best
# >> WR outcome, le pari ne tient que grâce au bonus, sensible au crowd Winamax).
# NB : E[pts 1/2/3] (barème simple) reste calculé dans df_m et résumé dans le Top 3
# ci-dessous, mais est retiré du tableau détaillé pour l'alléger.
#
# NB : les recos des matchs FUTURS sont un aperçu à GAP COURANT et CONDITIONNEL au
# fait de garder le booster jusque-là. Re-lance avec les gaps à jour quand le match
# devient courant.
import pandas as pd

# Noms des équipes par match (clé = position dans CDM_2026.csv + 1 = clé night_markets)
_cdm = pd.read_csv(DATA_DIR / "CDM_2026.csv")
match_labels = {
    i + 1: f"{str(r.team_A).replace('_', ' ')} - {str(r.team_B).replace('_', ' ')}"
    for i, r in enumerate(_cdm.itertuples(index=False))
}


def _afficher_marche(match_id, reco_m, wr_m, df_m):
    if HAS_BOOSTER:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m[["WR keep (%)", "WR x2 (%)"]].max(axis=1)
    else:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m["WR base (%)"]
    view = df_m.sort_values("WR best (%)", ascending=False).reset_index(drop=True)
    # E[pts 1/2/3] reste calculé dans df_m (cf. Top 3) mais retiré de l'affichage détaillé.
    view = view.drop(columns=["E[pts 1/2/3]"])

    fmt = {}
    for c in view.columns:
        if c.endswith("(%)") or c.startswith("E["):
            fmt[c] = "{:.2f}"
    fmt["Bonus"] = "{:.0f}"

    label = match_labels.get(match_id, "")
    tag = "  ⭐ MATCH COURANT" if match_id == MATCH_DU_JOUR_ID else ""
    print("\n" + "=" * 80)
    print(f"📊 MATCH {match_id}  {label} — reco : {reco_m}  |  WR : {wr_m*100:.2f}%{tag}")
    print("=" * 80)
    display(view.style.format(fmt).background_gradient(subset=["WR best (%)"], cmap="Greens"))

    agg = df_m.groupby("Outcome")["True Proba (%)"].sum().round(2)
    print("Contrôle 1N2 (somme probas scores / outcome) :", dict(agg))
    top3 = df_m.sort_values("E[pts 1/2/3]", ascending=False).head(3)
    tops = " | ".join(f"{r['Score']} ({r['E[pts 1/2/3]']:.3f})" for _, r in top3.iterrows())
    print(f"🏅 Top 3 E[pts 1/2/3] : {tops}")


for match_id, (reco_m, wr_m, df_m) in night_markets.items():
    _afficher_marche(match_id, reco_m, wr_m, df_m)


📊 MATCH 38  belgique - iran — reco : 2-0 (Safe)  |  WR : 65.49%  ⭐ MATCH COURANT


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,2-0,1 (Dom),13.64,15.43,50,33.99,58.66,65.49,62.09,64.83,65.49
1,1-0,1 (Dom),12.38,6.93,50,33.36,58.59,65.42,61.96,64.83,65.42
2,2-1,1 (Dom),10.10,18.53,50,32.22,58.48,65.32,61.76,64.83,65.32
3,3-0,1 (Dom),8.98,12.06,50,31.66,58.42,65.26,61.65,64.83,65.26
4,3-1,1 (Dom),7.07,16.29,50,30.70,58.32,65.17,61.47,64.83,65.17
5,4-0,1 (Dom),4.61,8.20,50,29.47,58.19,65.05,61.24,64.83,65.05
6,3-2,1 (Dom),2.68,2.33,70,29.04,58.15,65.01,61.14,64.83,65.01
7,4-1,1 (Dom),3.60,6.05,50,28.97,58.14,65.01,61.14,64.83,65.01
8,5-0,1 (Dom),1.91,4.24,70,28.51,58.09,64.96,61.05,64.83,64.96
9,5-1,1 (Dom),1.47,3.63,70,28.20,58.06,64.93,60.99,64.83,64.93


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(69.67), '2 (Ext)': np.float64(10.81), 'N (Nul)': np.float64(19.53)}
🏅 Top 3 E[pts 1/2/3] : 1-0 (1.075) | 2-0 (1.054) | 2-1 (1.052)

📊 MATCH 39  uruguay - cap vert — reco : 1-0 (Safe)  |  WR : 65.48%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,1-0,1 (Dom),18.33,20.28,30,54.01,58.65,65.48,63.95,64.93,65.48
1,3-0,1 (Dom),8.69,11.79,50,52.86,58.53,65.36,63.71,64.93,65.36
2,2-1,1 (Dom),8.44,19.90,50,52.73,58.51,65.35,63.69,64.93,65.35
3,2-0,1 (Dom),16.01,32.88,20,51.72,58.41,65.26,63.55,64.93,65.26
4,4-0,1 (Dom),3.95,3.36,70,51.28,58.36,65.21,63.40,64.93,65.21
5,3-1,1 (Dom),5.05,5.68,50,51.04,58.34,65.19,63.39,64.93,65.19
6,4-1,1 (Dom),2.28,1.57,70,50.11,58.24,65.10,63.21,64.93,65.10
7,3-2,1 (Dom),1.47,1.38,70,49.55,58.19,65.04,63.12,64.93,65.04
8,5-0,1 (Dom),1.40,1.59,70,49.50,58.18,65.04,63.11,64.93,65.04
9,4-2,1 (Dom),0.62,0.06,100,49.14,58.14,65.00,63.04,64.93,65.00


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(67.38), '2 (Ext)': np.float64(10.01), 'N (Nul)': np.float64(22.61)}
🏅 Top 3 E[pts 1/2/3] : 1-0 (1.139) | 2-0 (1.051) | 2-1 (1.041)

📊 MATCH 40  nouvelle zelande - egypte — reco : 0-1 (Safe)  |  WR : 65.48%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,0-1,2 (Ext),13.55,19.02,50,42.77,58.65,65.48,62.81,64.83,65.48
1,0-2,2 (Ext),12.21,25.57,30,39.66,58.34,65.19,62.28,64.83,65.19
2,0-3,2 (Ext),7.37,8.27,50,39.68,58.33,65.18,62.25,64.83,65.18
3,1-2,2 (Ext),9.50,28.38,30,38.84,58.25,65.11,62.13,64.83,65.11
4,1-3,2 (Ext),5.81,10.23,50,38.90,58.25,65.11,62.11,64.83,65.11
5,0-4,2 (Ext),3.38,2.03,70,38.36,58.19,65.05,61.99,64.83,65.05
6,1-4,2 (Ext),2.63,1.68,70,37.84,58.14,65.01,61.91,64.83,65.01
7,2-3,2 (Ext),2.30,1.53,70,37.61,58.12,64.98,61.87,64.83,64.98
8,0-5,2 (Ext),1.21,0.24,100,37.20,58.08,64.94,61.78,64.83,64.94
9,1-5,2 (Ext),0.93,0.11,100,36.92,58.05,64.92,61.74,64.83,64.92


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(15.37), '2 (Ext)': np.float64(61.01), 'N (Nul)': np.float64(23.63)}
🏅 Top 3 E[pts 1/2/3] : 0-1 (1.001) | 1-2 (0.961) | 0-2 (0.923)

📊 MATCH 41  argentine - autriche — reco : 1-0 (Safe)  |  WR : 65.48%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,1-0,1 (Dom),12.45,6.81,50,45.21,58.65,65.48,63.06,64.89,65.48
1,2-0,1 (Dom),11.68,19.47,50,44.82,58.61,65.45,63.00,64.89,65.45
2,3-0,1 (Dom),7.47,8.57,50,42.72,58.40,65.24,62.62,64.89,65.24
3,3-1,1 (Dom),6.10,16.97,50,42.03,58.33,65.18,62.50,64.89,65.18
4,2-1,1 (Dom),9.62,25.69,30,41.87,58.31,65.17,62.50,64.89,65.17
5,4-0,1 (Dom),3.66,3.69,70,41.54,58.27,65.13,62.39,64.89,65.13
6,4-1,1 (Dom),2.97,4.82,70,41.06,58.23,65.08,62.31,64.89,65.08
7,3-2,1 (Dom),2.63,4.45,70,40.83,58.20,65.06,62.27,64.89,65.06
8,5-0,1 (Dom),1.44,1.99,70,39.99,58.12,64.98,62.13,64.89,64.98
9,4-2,1 (Dom),1.24,4.43,70,39.85,58.10,64.97,62.11,64.89,64.97


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(61.88), '2 (Ext)': np.float64(14.33), 'N (Nul)': np.float64(23.79)}
🏅 Top 3 E[pts 1/2/3] : 1-0 (0.993) | 2-1 (0.965) | 2-0 (0.926)


In [8]:
# ==========================================
# 2. PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
# ==========================================
print("\n" + "="*50)
print("🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS")
print("="*50)
print("Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?\n")

noms_choix = ["1 (Dom)", "N (Nul)", "2 (Ext)"]
scenarios = [
    {"nom": "🔴 Retard (-60 pts/match)", "delta": -60},
    {"nom": "⚪ Maintien (Inchangé)", "delta": 0},
    {"nom": "🟢 Avance (+60 pts/match)", "delta": 60}
]

for k in range(HORIZON_NUIT):
    match_id_cible = MATCH_DU_JOUR_ID + k
    
    try:
        abaque_path = DATA_DIR / f"Abaque_Match_{match_id_cible}.npz"
        data = np.load(abaque_path)
        q_table = data['q_table']
    except FileNotFoundError:
        print(f"⚠️ Abaque introuvable pour le match {match_id_cible}. Fin de la projection.")
        break
        
    print(f"▶️ MATCH {match_id_cible} (Δt = +{k}) :")
    
    for sc in scenarios:
        proj_gap1 = MON_GAP_1 + (sc["delta"] * k)
        proj_gap2 = MON_GAP_2 + (sc["delta"] * k)
        
        g1_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap1)))) + GAP_OFFSET
        g2_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap2)))) + GAP_OFFSET
        
        if HAS_BOOSTER:
            wr_keep = q_table[g1_idx, g2_idx, :, 1]
            wr_use = q_table[g1_idx, g2_idx, :, 2]
            best_keep_idx, best_use_idx = np.argmax(wr_keep), np.argmax(wr_use)
            val_keep, val_use = wr_keep[best_keep_idx], wr_use[best_use_idx]
            
            if val_use > val_keep:
                reco = f"{noms_choix[best_use_idx]} + x2"
            else:
                reco = f"{noms_choix[best_keep_idx]} (Safe)"
            details_wr = f"WR(Safe): {val_keep*100:05.2f}% | WR(x2): {val_use*100:05.2f}%"
        else:
            wr_base = q_table[g1_idx, g2_idx, :, 0]
            best_action = np.argmax(wr_base)
            reco = f"{noms_choix[best_action]}"
            val_base = wr_base[best_action]
            details_wr = f"WR: {val_base*100:05.2f}%"
            
        print(f"  {sc['nom']:<27} | Gaps proj. : {proj_gap1:>4}, {proj_gap2:>4} | Reco : {reco:<14} | {details_wr}")
    print("-" * 90)


🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?

▶️ MATCH 38 (Δt = +0) :
  🔴 Retard (-60 pts/match)    | Gaps proj. : -137,   95 | Reco : 1 (Dom) (Safe) | WR(Safe): 65.23% | WR(x2): 61.22%
  ⚪ Maintien (Inchangé)       | Gaps proj. : -137,   95 | Reco : 1 (Dom) (Safe) | WR(Safe): 65.23% | WR(x2): 61.22%
  🟢 Avance (+60 pts/match)    | Gaps proj. : -137,   95 | Reco : 1 (Dom) (Safe) | WR(Safe): 65.23% | WR(x2): 61.22%
------------------------------------------------------------------------------------------
▶️ MATCH 39 (Δt = +1) :
  🔴 Retard (-60 pts/match)    | Gaps proj. : -197,   35 | Reco : 1 (Dom) (Safe) | WR(Safe): 59.17% | WR(x2): 57.12%
  ⚪ Maintien (Inchangé)       | Gaps proj. : -137,   95 | Reco : 1 (Dom) (Safe) | WR(Safe): 65.21% | WR(x2): 63.22%
  🟢 Avance (+60 pts/match)    | Gaps proj. :  -77,  155 | Reco : 1 (Dom) (Safe) | WR(Safe): 70.91% | WR(x2): 68.90%
-----------------------------